In [ ]:
# General packages.
import os
import warnings
import random
import torch
import glob
import time
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter

# Add tools path for additional Python code. 
import sys
sys.path.append("./tools/")

# Text processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Topic modelling
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from top2vec import Top2Vec
from bertopic import BERTopic
from Matave import Matave

In [ ]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)
# Ignore UserWarnings.
warnings.filterwarnings("ignore", category=UserWarning)
# Initialize constant variables.
INPUT_FOLDER = 'data'
# Topic Ranges
K_RANGE = list(range(3, 20))
TOP_N = 10

In [ ]:
stop_words = set(stopwords.words("english"))

## Text Processing

In [ ]:
# Make function to remove punctuation, make lowercase, remove stopwords, punctuation, remove documents with less than or equal to 1 token.
def preprocessing(notes, min_words=1):
    cleaned_notes = []

    for note in notes:
        tokens = word_tokenize(note, language='english')
        tokens = [token.lower() for token in tokens]
        tokens = [token for token in tokens if token.isalpha() and token not in stop_words]

        if len(tokens) >= min_words:
            cleaned_notes.append(" ".join(tokens))

    return cleaned_notes

In [ ]:
# Process all T1 files (this can be adjusted later for T2 also).
all_text_notes = {}
for folder in os.listdir(f'./{INPUT_FOLDER}/'):
    if '.' not in folder:
        for sub_folder in os.listdir(f'./{INPUT_FOLDER}/{folder}'):
            if '.' not in sub_folder and 'T1' in sub_folder:
                for sub_sub_folder in os.listdir(f'./{INPUT_FOLDER}/{folder}/{sub_folder}'):
                    if '.' not in sub_sub_folder:
                        carer_notes, carer_files = [], glob.glob(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/carerNotes*.xlsx')
                        nurse_notes, nurse_files = [], glob.glob(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/dailyNurseNotes*.xlsx')
                        multi_notes, multi_files = [], glob.glob(f'./{INPUT_FOLDER}/{folder}/{sub_folder}/{sub_sub_folder}/multiDisciplinaryNotes*.xlsx')
                        for carer_file in carer_files:
                            temp_df = pd.read_excel(f'{carer_file}', skiprows= 3, header=[0, 1])
                            carer_notes.extend([temp_note[0] for temp_note in temp_df['Activity'].dropna().values.tolist()])
                        for nurse_file in nurse_files:
                            temp_df = pd.read_excel(f'{nurse_file}')
                            nurse_notes.extend(temp_df['Note'].dropna().values.tolist())
                        for multi_file in multi_files:
                            temp_df = pd.read_excel(f'{multi_file}')
                            multi_notes.extend(temp_df['Note'].dropna().values.tolist())
                        all_text_notes[sub_sub_folder.split(' ')[0]] = {'Carer Notes': preprocessing(carer_notes), 'Multidisciplinary Notes': preprocessing(multi_notes), 'Nurse Notes': preprocessing(nurse_notes)}

In [ ]:
def count_notes(category):
    notes = []
    for key in all_text_notes:
        notes.extend([temp_text for temp_text in all_text_notes[key][category]])
    return pd.DataFrame.from_dict(Counter(notes), orient='index', columns=["count"])

In [ ]:
# Carer Notes are just category-like entries. Not free text.
count_notes('Carer Notes')

In [ ]:
# Multidisciplinary Notes are few but free text.
count_notes('Multidisciplinary Notes')

In [ ]:
# NUrse Notes are free text.
count_notes('Nurse Notes')

## Topic Modelling

In [ ]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

### LDA

In [ ]:
def lda_analysis(tokenized_texts, dictionary, corpus, dataset_name):
    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    
    for k in K_RANGE:
        start = time.time()

        lda_model = LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=RANDOM_STATE,
            passes=10
        )

        lda_topics = [
            [word for word, _ in lda_model.show_topic(i, topn=TOP_N)]
            for i in range(k)
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'LDA (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(lda_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(lda_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(lda_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(lda_topics)

    def normalize(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-9)

    norm_coh = normalize(np.array(metric_results['Coherence']))
    norm_div = normalize(np.array(metric_results['Diversity']))
    norm_red = normalize(np.array(metric_results['Redundancy']))

    w_coh = 1
    w_div = 1
    w_red = 1

    composite_scores = (
        w_coh * norm_coh +
        w_div * norm_div +
        w_red * norm_red
    )

    best_index = np.argmax(composite_scores)

    print(f"Coherence: {metric_results['Coherence'][best_index]}")
    print(f"Diversity: {metric_results['Diversity'][best_index]}")
    print(f"Inverse Redundancy: {metric_results['Redundancy'][best_index]}")
    print(f"Time (seconds): {metric_results['Time'][best_index]}")

    temp_top_row = metric_results['Top Topic Words'][best_index]
    print("----- Cluster Topics -----")
    for topic in temp_top_row:
        print(topic)
    print(f"Number of Topics: {len(temp_top_row)}")
    return best_index, metric_results

### NMF

In [ ]:
def nmf_analysis(texts, tokenized_texts, dictionary, dataset_name):
    # Vectorize texts for NMF.
    vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=2,
    stop_words='english'
    )
    tfidf = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()

    metric_results = {'Dataset Name': [], 'Algorithm Name': [], 'Coherence': [], 'Diversity': [], 'Redundancy': [], 'Time': [], 'Top Topic Words': []}
    for k in K_RANGE:
        start = time.time()

        nmf_model = NMF(
            n_components=k,
            random_state=RANDOM_STATE
        )
        nmf_model.fit(tfidf)

        nmf_topics = [
            [feature_names[i] for i in topic.argsort()[:-TOP_N - 1:-1]]
            for topic in nmf_model.components_
        ]

        end = time.time()

        metric_results['Algorithm Name'].append(f'NMF (K={k})')
        metric_results['Dataset Name'].append(dataset_name)
        metric_results['Coherence'].append(
            get_coherence_score(nmf_topics, tokenized_texts, dictionary, 'c_v')
        )
        metric_results['Diversity'].append(get_diversity_score(nmf_topics))
        metric_results['Redundancy'].append(compute_topic_redundancy(nmf_topics))
        metric_results['Time'].append(end - start)
        metric_results['Top Topic Words'].append(nmf_topics)

    def normalize(x):
            return (x - x.min()) / (x.max() - x.min() + 1e-9)

    norm_coh = normalize(np.array(metric_results['Coherence']))
    norm_div = normalize(np.array(metric_results['Diversity']))
    norm_red = normalize(np.array(metric_results['Redundancy']))

    w_coh = 1
    w_div = 1
    w_red = 1

    composite_scores = (
        w_coh * norm_coh +
        w_div * norm_div +
        w_red * norm_red
    )

    best_index = np.argmax(composite_scores)

    print(f"Coherence: {metric_results['Coherence'][best_index]}")
    print(f"Diversity: {metric_results['Diversity'][best_index]}")
    print(f"Inverse Redundancy: {metric_results['Redundancy'][best_index]}")
    print(f"Time (seconds): {metric_results['Time'][best_index]}")

    temp_top_row = metric_results['Top Topic Words'][best_index]
    print("----- Cluster Topics -----")
    for topic in temp_top_row:
        print(topic)
    print(f"Number of Topics: {len(temp_top_row)}")
    return best_index, metric_results

### Top2Vec

In [ ]:
def top2vec_analysis(texts, tokenized_texts, dictionary):
    start = time.time()
    top2vec_model = Top2Vec(
        texts,
        embedding_model='all-MiniLM-L6-v2',
        speed="learn"
    )
    cluster_topics = (top2vec_model.get_topics())[0]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

### BERTopic

In [ ]:
def bertopic_analysis(texts, tokenized_texts, dictionary):
    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

### MATAVE

In [ ]:
def matave_analysis(texts, tokenized_texts, dictionary, file_name = ""):
    # MATAVE
    start = time.time()
    matave = Matave(texts)
    matave.fit(k_range = K_RANGE)
    cluster_topics = [topic.split() for topic in matave.top_topic_words.values()]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

In [ ]:
all_nurse_notes = []
for patient in all_text_notes:
    texts = all_text_notes[patient]['Nurse Notes']
    all_nurse_notes.extend(texts)
    # Prepare components for evaluation.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)
    corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

    # lda
    print("----------- LDA -----------")
    lda_analysis(tokenized_texts, dictionary, corpus, patient)
    # nmf
    print("----------- NMF -----------")
    nmf_analysis(texts, tokenized_texts, dictionary, patient)
    # top2vec
    print("----------- Top2Vec -----------")
    top2vec_analysis(texts, tokenized_texts, dictionary)
    # bertopic
    print("----------- BERTopic -----------")
    bertopic_analysis(texts, tokenized_texts, dictionary)
    # matave
    print("----------- MATAVE -----------")
    matave_analysis(texts, tokenized_texts, dictionary)

In [ ]:
# Prepare components for evaluation.
tokenized_all_notes = [word_tokenize(text.lower()) for text in all_nurse_notes]
dictionary_all_notes = Dictionary(tokenized_all_notes)
corpus_all_notes = [dictionary.doc2bow(text) for text in tokenized_all_notes]

# lda
print("----------- LDA -----------")
lda_analysis(tokenized_all_notes, dictionary_all_notes, corpus_all_notes, "All Notes")
# nmf
print("----------- NMF -----------")
nmf_analysis(all_nurse_notes, tokenized_all_notes, dictionary_all_notes, "All Notes")
# top2vec
print("----------- Top2Vec -----------")
top2vec_analysis(all_nurse_notes, tokenized_all_notes, dictionary_all_notes)
# bertopic
print("----------- BERTopic -----------")
bertopic_analysis(all_nurse_notes, tokenized_all_notes, dictionary_all_notes)
# matave
print("----------- MATAVE -----------")
matave_analysis(all_nurse_notes, tokenized_all_notes, dictionary_all_notes)